In [1]:
import sys,os
from datetime import datetime
from pathlib import Path
import json
import torch # 张量、自动求导等
import torch.nn as nn # 网络层，Conv1D、LSTM等
import torch.nn.functional as F # 函数层， ReLU、cross_entropy等
from pyexpat import features
from torchinfo import summary
from torch.utils.data import DataLoader,TensorDataset
import time
import gc
from sklearn.metrics import roc_auc_score
import re
import math
import mne
from zmq import DEALER

from train import learning_rate, hidden_size
import numpy as np

root = Path(r"C:\Users\hank\Desktop\Research Reproduction\dianxian\self")
chb_data_path = Path(r"D:\data\dianxian\channal_select\chb03")

if str(root) not in sys.path:
    sys.path.append(str(root))

import tools as t

log_dir = root / "train_log"
embeddings = root / 'embeddings'
weight = root / 'weight'
Fp1_F7 = embeddings / 'Fp1-F7'
Fp2_F8 = embeddings / 'Fp2-F8'
T7_P7 = embeddings / 'T7-P7'
T8_P8 = embeddings / 'T8-P8'
P7_O1 = embeddings / 'P7-O1'
P8_O2 = embeddings / 'P8-O2'

seed = 1
val = (0.2,0.1)
method = "LOOCV"
out_channals = 32
batch = 32
elec = 6
embed_dim = 32
num_heads = 2
window_length = 10 * 256
window_overlap = 5 * 256
elec_names = ['Fp1-F7', 'Fp2-F8', 'T7-P7', 'T8-P8', 'P7-O1', 'P8-O2']

In [2]:
def get_positions(elec_names,montage_name="standard_1020"):
    montage = mne.channels.make_standard_montage(montage_name)
    ch_list, dig_list = montage.ch_names, montage.dig
    pos = []
    for name in elec_names:
        a, b = name.split("-")
        # standard_1020 里是 'Fp1'、'F7'，注意大小写
        va = np.array(dig_list[ch_list.index(a)]['r'])
        vb = np.array(dig_list[ch_list.index(b)]['r'])
        pos.append((va + vb) / 2.0)
    return torch.tensor(np.stack(pos), dtype=torch.float32) # (6,3)

def build_adjacency(elec_names):
    """距离倒数邻接矩阵，对角线为 0"""
    p = get_positions(elec_names)
    n = len(elec_names)
    adj = torch.zeros(n, n)
    for i in range(n):
        for j in range(n):
            if i != j:
                d = torch.norm(p[i] - p[j])
                if d > 0:
                    adj[i, j] = 1.0 / d
    return adj  # (6, 6)

In [3]:
class GraphConstructor(nn.Module):
    def __init__(self, num_nodes=6, features=32, hidden_dim=128, pos_adj=build_adjacency(elec_names)):
        # features：卷积输出的特征数
        # hidden_size：动态图隐藏层维度
        super().__init__()
        self.num_nodes = num_nodes
        # 全局静态图
        self.static_weights = nn.Parameter(torch.eye(num_nodes))
        # 打分MLP，最后一维是 (B,L,N,N) 的 j 维，归一化必须在 dim=3
        self.similarity_net = nn.Sequential(
            nn.Linear(features*2, hidden_dim),
            nn.ELU(inplace=False),
            nn.Linear(hidden_dim, 1),
            nn.Softmax(dim=3))
        # 物理图权重
        self.pos_factor = nn.Parameter(torch.tensor(0.5))
        # 冻结物理图，不参与训练
        # 常量
        self.register_buffer("pos_encoding", pos_adj.clone())

    # 局部动态图
    def compute_similarity(self, x): # x: (B, C, 6, L/5)
        B, D, N, L = x.shape
        x = x.reshape(B,L,N,D)
        node_i = x.unsqueeze(2).expand(-1, -1, N, -1, -1)
        node_j = x.unsqueeze(3).expand(-1, -1, -1, N, -1)
        # 按特征维拼接
        pairs = torch.cat([node_i, node_j], dim=-1) # (B, L, N, N, 2D)
        # 打分
        return self.similarity_net(pairs).squeeze(-1) # (B, L, N, N)

    def forward(self, x): # x: (B, C, 6, L/5)
        # 对特征维归一化
        static_adj = F.softmax(self.static_weights, dim=1)
        dynamic_adj = self.compute_similarity(x)
        pos_adj = F.softmax(self.pos_encoding, dim=1)
        return F.softmax(static_adj + dynamic_adj + self.pos_factor * pos_adj, dim=3) # (B,L,N,N)

    # adjacency的列与x的行保持一致，保证正常矩阵乘法
    # 由于是每一行的所有列在对x做变换，因此对dim=3(列)做归一化

C:\Users\hank\AppData\Local\Temp\ipykernel_30176\4103429691.py:2: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage(montage_name)


In [4]:
class GraphConvolution(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_dim=128):
        """
        in_channels：输入节点特征维度
        out_channels：输出节点特征维度
        hidden_dim：构建图时的隐藏层维度
        """
        super().__init__()

        # 打分层要按输入特征维度拼接，所以 features 必须跟着 in_channels 走
        self.gc = GraphConstructor(features=in_channels, hidden_dim=hidden_dim)

        # 对每个节点的特征进行线性变换
        self.fc = nn.Linear(in_channels, out_channels)

    def forward(self, x): # x: (B, C, N, L)
        B, C, N, L = x.shape
        # 1. 图构建
        adjacency = self.gc(x)  # (B, L, N, N)
        # 2. 调整节点特征维度
        x = x.permute(0, 3, 2, 1)  # (B, L, N, C)
        # 3. 合并批次维度和时间维度
        adjacency = adjacency.reshape(B * L, N, N)
        x = x.reshape(B * L, N, C)
        # 4. 空间聚合
        x = torch.bmm(adjacency, x)  # (B*L, N, C)
        # 5. 特征变换
        x = self.fc(x)  # (B*L, N, out_channels)
        # 6. 恢复维度
        x = x.reshape(B, L, N, -1)
        x = x.permute(0, 3, 2, 1)
        return x  # (B, out_channels, N, L)

In [5]:
from models import CommonConv2d,BILSTMClassify

In [6]:
# m = CommonConv2d()
# summary(m,(32,1,6,2500)) # (32, 32, 6, 78)

In [7]:
class Detection(nn.Module):
    def __init__(self,in_channals=1,out_channals=32,embed_dim=16,num_heads=2,temporal_attention=False,channal_attention=False,dropout=0):
        super().__init__()
        # 输入是 CommonConv2d 出来的 (B,32,6,L/32)，所以第一层 in_channels=32
        self.gcn = nn.Sequential(
            GraphConvolution(in_channels=32,out_channels=16,hidden_dim=64),
            GraphConvolution(in_channels=16,out_channels=32,hidden_dim=128)
        )
        self.conv2d = CommonConv2d()
        self.Bilstm = BILSTMClassify(
            temporal_attention = temporal_attention,
            channal_attention = channal_attention,
            dropout=dropout
        )
        # 图卷积与卷积融合
        self.fc = nn.Linear(out_channals*2,out_channals)
        # self.atten = nn.MultiheadAttention(embed_dim, num_heads=num_heads, batch_first=True)

    def forward(self,x): # (B,6,L)
        x1 = x.unsqueeze(1) # (B,1,6,L)
        # 卷积分支
        conv_feat = self.conv2d(x1) # (B, 32, 6, 78)
        # 图卷积分支
        gcn_feat = self.gcn(conv_feat) # (B, 32, 6, 78)
        x = torch.cat((gcn_feat, conv_feat), dim=1) # (B, 32*2, 6, 78)
        x = x.permute(0, 2, 3, 1)  # (B, 6, 78, 64)
        x = self.fc(x) # (B, 6, 78, 32)

        # 时序预测，每个电极一个 logit，再对六通道取平均
        B, N, L, C = x.shape
        x = x.permute(0,2,3,1).reshape(B*N,L,C)
        x = self.Bilstm(x) # (B*6, 1)
        return x.reshape(B,N).mean(dim=1,keepdim=True) # (B, 1)

In [8]:
from datetime import datetime

In [9]:
windows,meta = t.preprocess(r"D:\data\dianxian\wang2021\chb22",SPH=2*60,SOP=30*60,interictal_thr=2*60*60,window_length=4,window_overlap=2,elec_names=elec_names)

[边界] chb22 末尾不足整窗: ['chb22_11.edf(丢 1.0s)']
[chb22] 原始窗 55774
[chb22]   label=0 39544
[chb22]   label=1 2366
[chb22]   label=2 96
[chb22]   label=3 13768
[chb22]   excluded(overlap) 22
[chb22]   excluded(sph) 174
[chb22]   excluded(exclusion) 13572
[chb22] usable 42006 / excluded 13768
[chb22] Fp1-F7 窗 55774
[chb22] Fp2-F8 窗 55774
[chb22] T7-P7 窗 55774
[chb22] T8-P8 窗 55774
[chb22] P7-O1 窗 55774
[chb22] P8-O2 窗 55774


In [10]:
teacher_data_set = t.teacher_train_val_test(elec_names, meta, windows, seed, val=(0.1, 0.1), method="LOOCV")
student_data_set = t.train_val_test(elec_names, meta, windows, seed, val=(0.1, 0.2), method="LOOCV")

[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       34/34          35/35          27/27          54078/0         
[划分] S1       34/34          27/27          35/35          54070/0         
[划分] S2       27/27          35/35          34/34          50143/0         
[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] metho

In [14]:
# TODO:Teacher中lr=0.001时fold3、fold1都很高，0.00001时，fold2很高，可能要自适应学习率

# 每折的 train/val/test -> DataLoader
def teacher_loaders(data_set,fold,batch_size=batch):
    loaders = []
    for key in ("train","val","test"):
        loaders.append(DataLoader(TensorDataset(data_set[fold][key+"_wins"],data_set[fold]["y_"+key]),
                                  batch_size=batch_size,shuffle=(key=="train")))
    return loaders


# 教师每折重训一次（LOOCV），六通道一起喂
teacher_result = {}
optimizer_lr = 0.00001
loss_fuction = nn.BCEWithLogitsLoss() # 自带sigmoid()
epochs = 50
patience = 5
now = datetime.now().strftime("%y%m_%d_%H%M%S")

for fold in range(len(teacher_data_set)):
    teacher = Detection().to("cuda")
    optimizer = torch.optim.Adam(teacher.parameters(),lr=optimizer_lr)
    teacher_train_dataloader,teacher_val_dataloader,teacher_test_dataloader = teacher_loaders(teacher_data_set,fold)

    best_train_loss = np.inf
    best_state_dict = {}
    patience_count = 0
    print(f"===== fold {fold+1} seizure {teacher_data_set[fold]['seizure_id']} =====")

    for epoch in range(epochs):
        avg_train_loss = 0
        avg_val_loss = 0

        teacher.train()
        for (x,y) in teacher_train_dataloader:
            x,y = x.to("cuda"),y.to("cuda")
            optimizer.zero_grad()
            out = teacher(x)
            loss = loss_fuction(out, y)
            loss.backward()
            optimizer.step()
            avg_train_loss += loss.item()

        teacher.eval()
        with torch.no_grad():
            for (x,y) in teacher_val_dataloader:
                x,y = x.to("cuda"),y.to("cuda")
                out = teacher(x)
                loss = loss_fuction(out, y)
                avg_val_loss += loss.item()

        avg_train_loss /= len(teacher_train_dataloader)
        avg_val_loss /= len(teacher_val_dataloader)
        print(f"epoch:{epoch+1} avg_loss:{avg_train_loss:.4f} avg_val_loss:{avg_val_loss:.4f}")

        if avg_val_loss < best_train_loss:
            best_train_loss = avg_val_loss
            patience_count = 0
            best_state_dict = {k:v.cpu().clone() for k,v in teacher.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= patience:
                print("Early stop！！！")
                break

    # 用最好的权重测这一折的发作
    teacher.load_state_dict(best_state_dict)
    teacher.eval()
    teacher_targets,teacher_probs = [],[]
    with torch.no_grad():
        for (x,y) in teacher_test_dataloader:
            x,y = x.to("cuda"),y.to("cuda")
            teacher_probs.extend(torch.sigmoid(teacher(x)).cpu().numpy().flatten())
            teacher_targets.extend(y.cpu().numpy().flatten())

    teacher_auc = roc_auc_score(teacher_targets,teacher_probs)
    teacher_acc = ((np.array(teacher_probs) > 0.5) == np.array(teacher_targets)).mean()
    teacher_result[f"fold_{fold+1}"] = {"seizure_id":int(teacher_data_set[fold]["seizure_id"]),
                                        "n_test":len(teacher_targets),
                                        "auc":round(float(teacher_auc),4),
                                        "acc":round(float(teacher_acc),4)}
    print(f"fold:{fold+1} n:{len(teacher_targets)} AUC:{teacher_auc:.4f} ACC:{teacher_acc:.4f}")

    path = weight / f"{now}_teacher_fold{fold+1}.pth"
    torch.save(best_state_dict,path)

===== fold 1 seizure 0 =====
epoch:1 avg_loss:0.6961 avg_val_loss:0.7228
epoch:2 avg_loss:0.7092 avg_val_loss:0.7228
epoch:3 avg_loss:0.7114 avg_val_loss:0.7228
epoch:4 avg_loss:0.6963 avg_val_loss:0.7227
epoch:5 avg_loss:0.6821 avg_val_loss:0.7228
epoch:6 avg_loss:0.6812 avg_val_loss:0.7229
epoch:7 avg_loss:0.6828 avg_val_loss:0.7230
epoch:8 avg_loss:0.7201 avg_val_loss:0.7230
epoch:9 avg_loss:0.6942 avg_val_loss:0.7232
Early stop！！！
fold:1 n:54 AUC:0.5000 ACC:0.5000
===== fold 2 seizure 1 =====
epoch:1 avg_loss:0.6921 avg_val_loss:0.6930
epoch:2 avg_loss:0.6913 avg_val_loss:0.6929
epoch:3 avg_loss:0.6924 avg_val_loss:0.6928
epoch:4 avg_loss:0.6917 avg_val_loss:0.6928
epoch:5 avg_loss:0.6954 avg_val_loss:0.6926
epoch:6 avg_loss:0.6939 avg_val_loss:0.6925
epoch:7 avg_loss:0.6932 avg_val_loss:0.6924
epoch:8 avg_loss:0.6915 avg_val_loss:0.6923
epoch:9 avg_loss:0.6937 avg_val_loss:0.6922
epoch:10 avg_loss:0.6916 avg_val_loss:0.6921
epoch:11 avg_loss:0.6919 avg_val_loss:0.6920
epoch:12 avg

In [12]:
# teacher_train_dataloader
# teacher_val_dataloader
# teacher_test_dataloader
# teacher = Detection().to("cuda")
# optimizer = torch.optim.Adam(teacher.parameters(),lr=0.001)
# loss_fuction = nn.BCEWithLogitsLoss() # 自带sigmoid()
# epochs = 50
# best_train_loss = np.inf
# best_state_dict = {}
# patience = 5
# patience_count = 0
#
# for epoch in range(epochs):
#     avg_train_loss = 0
#     avg_val_loss = 0
#
#     teacher.train()
#     for (x,y) in teacher_train_dataloader:
#         x,y = x.to("cuda"),y.to("cuda")
#         optimizer.zero_grad()
#         out = teacher(x)
#         loss = loss_fuction(out, y)
#         loss.backward()
#         optimizer.step()
#         avg_train_loss += loss.item()
#
#
#     teacher.eval()
#     with torch.no_grad():
#         for (x,y) in teacher_val_dataloader:
#             x,y = x.to("cuda"),y.to("cuda")
#             out = teacher(x)
#             loss = loss_fuction(out, y)
#             avg_val_loss += loss.item()
#
#     print(f"avg_loss:{avg_train_loss}")
#     print(f"avg_val_loss:{avg_val_loss}")
#
#     if avg_val_loss < best_train_loss:
#         best_train_loss = avg_val_loss
#         patience_count = 0
#         best_state_dict = teacher.state_dict()
#     else:
#         patience_count += 1
#         if patience_count >= patience:
#             print("Early stop！！！")
#             break
#
# now = datetime.now().strftime("%y%m_%d_%H%M%S")
# path = Path(f"{now}_teacher.pth")
# torch.save(best_state_dict, weight / path)


